# RouteFlow AI — Cross-Border Payment Routing Pipeline Walkthrough
## Interactive Verification of Ingestion, Training, Decision Optimization, and API Serving

This notebook demonstrates the end-to-end Machine Learning and Decision Optimization pipeline for RouteFlow AI:
1. **Data Ingestion & Quality Validation:** Schema conformance, null rate checks, duplicate detection, and stratified data partitioning.
2. **Feature Engineering & Model Training:** Offline pipeline training calibrated GBDT classification (success probability) and log-transformed regression (settlement latency).
3. **Production vs. Exploratory Analysis:** Transitioning from research code to thread-safe, scalable production software.
4. **REST API Serving:** Programmatic invocation of FastAPI inference and route recommendation endpoints.
5. **Automated Quality Assurance:** Automated verification with pytest covering unit, integration, and property-based directional tests.
6. **Data & Model Quality Metrics:** Quantitative evaluations including ROC-AUC, calibration error (ECE), regression MAE, and covariate drift (PSI).
7. **Code Standards & Linting:** Strict compliance with Black, Ruff, and PEP 8.


## 1. Data Ingestion and Validation (Modular OOP Architecture) (Modular OOP Architecture)
We encapsulate data ingestion and validation into single-responsibility classes:
- `DataIngestion`: loads raw data with error handling and logging.
- `DataValidator`: validates schema, missing values, duplicate IDs, temporal consistency, and creates stratified train/val/test splits.


In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("src"))

from data_ingestion import DataIngestion, DataValidator

# 1. Ingest raw dataset
ingestor = DataIngestion("data.csv")
df = ingestor.load_data()
print("Dataset Shape:", df.shape)

# 2. Validate data quality
validator = DataValidator()
report = validator.validate(df)
print("\n=== Data Quality Report ===")
print("Total Rows:", report.rows, "Columns:", report.columns)
print("Schema Valid:", report.schema_ok)
print("Missing Target Values:", report.missing_targets)
print("Duplicate IDs:", report.duplicate_transaction_ids)
print("Temporal Consistency OK:", report.temporal_consistency_ok)
print("Amount Range:", report.amount_range)
print("Historical Success Rate Range OK:", report.success_rate_range_ok)

# 3. Create stratified splits
train_df, val_df, test_df = validator.create_splits(df)
print("\nStratified Split Sizes -> Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))


[2026-10-28 12:44:44,549] [INFO] [data_ingestion]: Ingesting raw dataset from: data.csv
[2026-10-28 12:44:44,664] [INFO] [data_ingestion]: Successfully ingested 50000 records with 20 columns
Dataset Shape: (50000, 20)
[2026-10-28 12:44:44,664] [INFO] [data_ingestion]: Starting data quality validation on dataframe with shape (50000, 20)
[2026-10-28 12:44:44,682] [WARNING] [data_ingestion]: Missing values detected in columns: {'payment_type': 500, 'historical_route_success_rate': 750, 'historical_route_failure_rate': 750, 'historical_avg_latency_seconds': 1000}
[2026-10-28 12:44:44,710] [INFO] [data_ingestion]: Validation completed. Schema OK: True, Rows: 50000

=== Data Quality Report ===
Total Rows: 50000 Columns: 20
Schema Valid: True
Missing Target Values: 0
Duplicate IDs: 0
Temporal Consistency OK: True
Amount Range: [10.0, 100000.0]
Historical Success Rate Range OK: True
[2026-10-28 12:44:44,710] [INFO] [data_ingestion]: Splitting 50000 records with ratio 0.70/0.15/0.15
[2026-10-28

## 2. Feature Engineering and Offline Model Training Pipeline
The offline component uses:
- `FeaturePipeline`: ColumnTransformer applying SimpleImputer, OneHotEncoder, and StandardScaler.
- `ModelTrainer`: Fits a calibrated `HistGradientBoostingClassifier` (Prediction Goal 1) and a log-target `HistGradientBoostingRegressor` (Prediction Goal 2), tunes the decision threshold on validation data, and persists artifacts.


In [2]:
import json
import pandas as pd
from model_trainer import ModelTrainer

trainer = ModelTrainer(artifacts_dir="artifacts", figures_dir="figures")
metrics = json.load(open("artifacts/metrics.json"))

print("=== Prediction Goal 1: Payment Success Classifier (Held-out Test Set) ===")
print(pd.DataFrame(metrics["classification"]).T.round(4).to_string())
print("\nTuned Decision Threshold (Validation):", round(metrics['decision_threshold'], 3))
print("Expected Calibration Error (ECE):", round(metrics['expected_calibration_error'], 4))

print("\n=== Prediction Goal 2: Settlement Time Regressor (Completed Test Payments) ===")
print(pd.DataFrame(metrics["regression_success_only"]).T.round(3).to_string())
print("\nModel Version:", metrics['model_version'])


=== Prediction Goal 1: Payment Success Classifier (Held-out Test Set) ===
                                  roc_auc   brier  log_loss  balanced_accuracy  failure_recall  accuracy  precision  recall      f1
gbdt_classifier                    0.6980  0.0761    0.2746             0.6388          0.6774    0.6069     0.9514  0.6002  0.7361
logistic_baseline                  0.6994  0.0760    0.2745             0.6454          0.6605    0.6329     0.9513  0.6303  0.7582
majority_baseline                  0.5000  0.0868    1.1992             0.5000          0.0000    0.9132     0.9132  1.0000  0.9546
historical_success_rate_baseline   0.6831  0.0768       NaN                NaN             NaN       NaN        NaN     NaN     NaN

Tuned Decision Threshold (Validation): 0.91
Expected Calibration Error (ECE): 0.0066

=== Prediction Goal 2: Settlement Time Regressor (Completed Test Payments) ===
                             mae_seconds  rmse_seconds     r2  mae_minutes
gbdt_regressor           

## 3. Production Architecture vs. Exploratory Prototype
We compare the experimental prototype approach vs. the modular production inference architecture:
- Research prototype in notebook: ad-hoc in-memory execution without error handling or thread safety.
- Production module (`src/inference.py`): Thread-safe Singleton `PaymentPredictor`, cached models, high-resolution latency measurement, domain range bounds clamping, and defensive validation.


In [3]:
from inference import PaymentPredictor

predictor = PaymentPredictor.get_instance()
print("Predictor Ready:", predictor.is_loaded)
print("Model Version:", predictor.metrics.get("model_version"))
print("Active Candidate Routes:", len(predictor.route_catalog))

sample_payment = {
    "transaction_id": "NB_DEMO_001",
    "source_country": "US",
    "destination_country": "India",
    "source_currency": "USD",
    "destination_currency": "INR",
    "amount": 2500.0,
    "payment_type": "P2P",
    "hour_of_day": 14,
    "day_of_week": 2,
    "is_weekend": 0
}

pred_res = predictor.predict(sample_payment)
print(f"\nGenerated predictions for {len(pred_res['predictions'])} routes in {pred_res['inference_ms']:.2f} ms:")
for p in pred_res['predictions']:
    hrs = p['predicted_settlement_seconds'] / 3600.0
    print(f"  Route {p['route_id']:8s} | P(success) = {p['predicted_success_probability']:.3f} | Settle = {hrs:.2f} h | Cost = {p['route_cost']:.2f}")


[2026-10-28 12:44:45,112] [INFO] [inference]: Loading model artifacts from: ./artifacts
[2026-10-28 12:44:45,162] [INFO] [inference]: Successfully loaded models: version=v20261028-121925, threshold=0.910, routes=6
Predictor Ready: True
Model Version: v20261028-121925
Active Candidate Routes: 6
[2026-10-28 12:44:45,203] [INFO] [inference]: Inference successful: 6 routes evaluated in 41.38 ms (version=v20261028-121925)

Generated predictions for 6 routes in 41.38 ms:
  Route ROUTE_A  | P(success) = 0.928 | Settle = 0.99 h | Cost = 1.50
  Route ROUTE_B  | P(success) = 0.979 | Settle = 0.08 h | Cost = 5.00
  Route ROUTE_C  | P(success) = 0.833 | Settle = 23.31 h | Cost = 0.50
  Route ROUTE_D  | P(success) = 0.914 | Settle = 1.95 h | Cost = 2.00
  Route ROUTE_E  | P(success) = 0.958 | Settle = 0.49 h | Cost = 3.50
  Route ROUTE_F  | P(success) = 0.875 | Settle = 11.67 h | Cost = 1.00


## 4. REST API (FastAPI) Endpoints & Live Invocations
The application exposes:
- `GET /health`: Service health and model status.
- `GET /model-info`: Safe metadata and quality metrics.
- `POST /predict`: Success probability and settlement latency.
- `POST /recommend-route`: Multi-objective optimal corridor selection with hard business constraints.
- Validation handling: Rejection of invalid types, negative amounts, or bad categories with HTTP 422.


In [4]:
from fastapi.testclient import TestClient
from api.app import app

client = TestClient(app)

# 1. Health check
res_health = client.get("/health")
print("1. GET /health ->", res_health.status_code, res_health.json())

# 2. Model info
res_info = client.get("/model-info")
print("\n2. GET /model-info ->", res_info.status_code, "Version:", res_info.json().get("model_version"))

# 3. Route Recommendation
req_payload = {
    "payment": {
        "transaction_id": "TXN_API_DEMO",
        "source_country": "UK",
        "destination_country": "Nigeria",
        "source_currency": "GBP",
        "destination_currency": "NGN",
        "amount": 50000.0,
        "payment_type": "B2B"
    },
    "mode": "optimisation",
    "weights": {"success": 0.6, "latency": 0.25, "cost": 0.15},
    "constraints": {"max_cost": 3.0, "min_success_probability": 0.85}
}
res_route = client.post("/recommend-route", json=req_payload)
route_data = res_route.json()
print("\n3. POST /recommend-route ->", res_route.status_code)
print(f"   Recommended Route: {route_data['recommended_route']}")
print(f"   Explanation: {route_data['explanation']}")
print(f"   Constraints Relaxed: {route_data['constraints_relaxed']}")

# 4. Input validation error test
res_bad = client.post("/predict", json={"payment": {"source_country": "US", "amount": -100.0, "payment_type": "INVALID"}})
print("\n4. POST /predict (Bad Input) -> HTTP", res_bad.status_code)
print("   Validation response:", res_bad.json())


1. GET /health -> 200 {'status': 'ok', 'model_loaded': True, 'service': 'payment-routing-api', 'timestamp': 1793171685.740156}

2. GET /model-info -> 200 Version: v20261028-121925
[2026-10-28 12:44:45,765] [INFO] [decision_engine]: Route recommendation determined: ROUTE_A (score=0.925, relaxed=False)

3. POST /recommend-route -> 200
   Recommended Route: ROUTE_A
   Explanation: ROUTE_A via Bank_A->Bank_X: P(success)=95.2%, expected settlement 1.05 h, cost 1.50. Selected because it has the best weighted trade-off of success, latency and cost.
   Constraints Relaxed: False
[2026-10-28 12:44:45,768] [WARNING] [api]: Validation error on POST /predict: ['body -> payment -> destination_country: Field required', 'body -> payment -> source_currency: Field required', 'body -> payment -> destination_currency: Field required', 'body -> payment -> amount: Input should be greater than 0', "body -> payment -> payment_type: Input should be 'P2P', 'B2B', 'C2B' or 'B2C'"]

4. POST /predict (Bad Input) 

## 5. Automated Quality Assurance with Pytest
We run the complete automated test suite comprising 25 tests:
- Data validation tests (`tests/test_data_validation.py`)
- Decision engine tests (`tests/test_decision_engine.py`)
- FastAPI endpoint tests (`tests/test_api.py`)
- Integration tests (`tests/test_integration.py`)
- **ML Training Test (Req 7a):** Overfitting on a small batch to verify loss decreases and learning occurs (`tests/test_model_training.py`).
- **ML Inference Tests (Req 7b):** Output ranges, probability bounds, directional monotonicity, and invariance (`tests/test_model_inference.py`).


In [5]:
import subprocess

res = subprocess.run(["pytest", "-v"], capture_output=True, text=True)
print(res.stdout)


============================= test session starts ==============================
platform darwin -- Python 3.14.2, pytest-9.1.1, pluggy-1.6.0 -- ./venv/bin/python3.14
cachedir: .pytest_cache
rootdir: .
configfile: pyproject.toml
testpaths: tests
plugins: platformdirs-4.13.7, anyio-4.15.1
collecting ... collected 25 items

tests/test_api.py::test_health_check_endpoint PASSED                     [  4%]
tests/test_api.py::test_model_info_endpoint PASSED                       [  8%]
tests/test_api.py::test_predict_endpoint_valid_request PASSED            [ 12%]
tests/test_api.py::test_predict_endpoint_custom_routes PASSED            [ 16%]
tests/test_api.py::test_recommend_route_endpoint PASSED                  [ 20%]
tests/test_api.py::test_recommend_route_with_constraints PASSED          [ 24%]
tests/test_api.py::test_api_validation_errors PASSED                     [ 28%]
tests/test_api.py::test_openapi_schema_availability PASSED               [ 32%]
tests/test_data_validation.py::test_

## 6. Data Quality and Model Quality Metrics Evaluation
We evaluate quantitative quality indicators:
- **Model Quality:** ROC-AUC ($0.6980$), F1 ($0.7361$), Brier score ($0.0761$), Expected Calibration Error ($0.66\%$), Regression MAE ($69.7$ min), and $R^2$ ($0.942$).
- **Data Quality:** Missing value rate, duplicate rate, range violations, and Population Stability Index (PSI) drift detection between baseline and incoming data.


In [6]:
from metrics_evaluator import DataQualityEvaluator, ModelQualityEvaluator

evaluator = DataQualityEvaluator(baseline_df=train_df)
dq_report = evaluator.evaluate(test_df)

print("=== Data Quality Metrics (Test Split) ===")
print("Missing Value Rate:", round(dq_report['missing_value_rate'], 4))
print("Duplicate Record Rate:", round(dq_report['duplicate_record_rate'], 4))
print("Feature Range Violation Rate:", round(dq_report['range_violation_rate'], 4))
print("\nDrift Assessment (Population Stability Index vs Training Baseline):")
for col, drift in dq_report['drift_assessment'].items():
    print(f"  {col:32s}: PSI = {drift['psi']:.4f} ({drift['status']})")


=== Data Quality Metrics (Test Split) ===
Missing Value Rate: 0.0031
Duplicate Record Rate: 0.0
Feature Range Violation Rate: 0.0

Drift Assessment (Population Stability Index vs Training Baseline):
  amount                          : PSI = 0.0011 (STABLE)
  hour_of_day                     : PSI = 0.0021 (STABLE)
  day_of_week                     : PSI = 0.0004 (STABLE)
  is_weekend                      : PSI = 0.0001 (STABLE)
  route_cost                      : PSI = 0.0001 (STABLE)
  historical_route_success_rate   : PSI = 0.0000 (STABLE)
  historical_avg_latency_seconds  : PSI = 0.0001 (STABLE)


## 7. Code Formatting & Linting Verification
Verifying that all Python modules and tests conform to `black` formatting and `ruff` lint rules configured in `pyproject.toml`.


In [7]:
import subprocess

black_out = subprocess.check_output(["black", "--check", "src", "tests"]).decode()
print("=== Black Format Check ===")
print(black_out)

ruff_out = subprocess.check_output(["ruff", "check", "src", "tests"]).decode()
print("\n=== Ruff Lint Check ===")
print(ruff_out)


=== Black Format Check ===

=== Ruff Lint Check ===
All checks passed!

